# 03. 1D-CNN Market Signal Encoder Architecture

**Objective:** Implement and verify the 1D Convolutional Neural Network (1D-CNN) Market Signal Encoder as specified in Section 3 of the research paper.

---

## 1. Architectural Blueprint
The paper specifies:
- **Embedding Layer:** $d = 100$ embedding dimensions with zero padding at index 0.
- **Parallel 1D Convolutions:** Kernels of size $k \in \{3, 4, 5\}$, with $100$ filters per kernel size ($300$ total pooled feature maps).
- **Non-Linearity & Pooling:** $\text{ReLU}$ activation followed by Global 1D Max Pooling.
- **Classification Head:** Dense layer with $128$ $\text{ReLU}$ units, Dropout ($p = 0.5$), and a Linear output unit with Sigmoid activation.


In [1]:
import sys
import torch
import torch.nn as nn

sys.path.append("..")
from src import config
from src.model import MarketSignalCNN

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")


PyTorch Version: 2.15.0+cpu
CUDA Available: False


## 2. Model Instantiation & Layer Verification


In [2]:
model = MarketSignalCNN(
    vocab_size=config.MAX_VOCAB_SIZE,
    embedding_dim=config.EMBEDDING_DIM,
    filter_sizes=config.CNN_FILTER_SIZES,
    num_filters=config.NUM_FILTERS,
    dense_units=config.DENSE_UNITS,
    dropout_rate=config.DROPOUT_RATE
)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")


MarketSignalCNN(
  (embedding): Embedding(25000, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(100, 100, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): Conv1d(100, 100, kernel_size=(4,), stride=(1,), padding=(2,))
    (2): Conv1d(100, 100, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=300, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)
Total Trainable Parameters: 2,658,957


## 3. Forward Pass & Output Shape Test
We verify that a batch of token IDs passes through convolutions, pooling, and classification layers to output valid logits and probabilities.


In [3]:
dummy_batch = torch.randint(0, config.MAX_VOCAB_SIZE, (4, config.MAX_SEQ_LENGTH))
logits = model(dummy_batch)
probs = model(dummy_batch, return_probs=True)

print(f"Input Shape:         {dummy_batch.shape}")
print(f"Output Logits Shape: {logits.shape}")
print(f"Sample Probabilities: {probs.detach().numpy()}")


Input Shape:         torch.Size([4, 250])
Output Logits Shape: torch.Size([4])
Sample Probabilities: [0.4928598  0.49704862 0.5028372  0.4991529 ]


## 4. Discussion: Dual-Branch vs. Market Branch
The full paper proposes a dual-branch network integrating historical sprint delivery records (BiLSTM + attention). Because the paper's three startup datasets are proprietary and unreleased, we genuinely implement the market signal branch and document this methodological boundary.
